# Search & Access biodiversity data
This notebook queries the LifeWatch catalog (https://metadatacatalogue.lifewatch.eu/), accesses the GBIF record of the selected resource, and saves its data locally in a CSV file.

In [ ]:
import requests
import pandas as pd

## Data Search

In [ ]:
url = (
    "https://metadatacatalogue.lifewatch.eu/srv/api/search/records/_search"
)

params = {}

payload = {
    "query": {
        "query_string": {
            "query": "species"
        }
    },
}

payload = {
    "query": {
        "match": {
            "resourceTitle": "species"
        }
    }
}

r = requests.post(
    url,
    params=params,
    json=payload,
    headers={"Accept": "application/json"}
)
data = r.json()

In [ ]:
hits = data["hits"]["hits"]

for hit in hits:
    record_id = hit["_id"]
    source = hit["_source"]

    print("\nID:", record_id)

    # titolo
    title = source.get("resourceTitleObject", {})
    print("Title:", title)

## Data Selection

#### ID: ac2350c3-b912-4ce6-89bb-c922e15ec2cc

##### Title: {'default': 'Amphipod species in European Marine waters'}

In [ ]:
uuid = "ac2350c3-b912-4ce6-89bb-c922e15ec2cc"

url = f"https://metadatacatalogue.lifewatch.eu/srv/api/records/{uuid}"

headers = {
    "accept": "application/json",
}

response = requests.get(url, headers=headers)
record = response.json()

In [ ]:
distribution = record["dataset"]["distribution"]
gbif_url = next(d["url"] for d in distribution if isinstance(d, dict) and d.get("onlineDescription")=="Access to GBIF record")
dataset_key = gbif_url.split("/")[-1]

## Metadata Extraction

In [ ]:
meta = requests.get(f"https://api.gbif.org/v1/dataset/{dataset_key}").json()
print(meta["title"])
print(meta["description"])
print(meta.get("license"))

## Data Access

In [ ]:
url = "https://api.gbif.org/v1/occurrence/search"

limit = 300
offset = 0

results = []

while True:
    r = requests.get(
        url,
        params={
            "datasetKey": dataset_key,
            "limit": limit,
            "offset": offset
        }
    )
    r.raise_for_status()

    data = r.json()

    results.extend(data["results"])

    print(f"Downloaded {len(results)} / {data['count']}")

    if data["endOfRecords"]:
        break

    offset += limit

df = pd.DataFrame(results)

cols = [
    "scientificName",
    "eventDate",
    "decimalLatitude",
    "decimalLongitude",
    "country"
]

df = df[[c for c in cols if c in df.columns]]

In [ ]:
df.to_csv("./amphipod_species_EU_marine_waters.csv", index=False)